In [1]:
import sys
import pyspark
from pyspark.sql import SparkSession
import re
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
import time
from statistics import median




In [2]:
print("Python:", sys.version)
print("PySpark:", pyspark.__version__)

spark = SparkSession.builder.appName("benchmark").master('local[*]').getOrCreate()

print("Spark:", spark.version)

print(
    "AQE enabled:",
    spark.conf.get("spark.sql.adaptive.enabled")
)

print(
    "Shuffle partitions:",
    spark.conf.get("spark.sql.shuffle.partitions")
)

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PySpark: 4.0.4
Spark: 4.0.4
AQE enabled: true
Shuffle partitions: 200


In [3]:


N = 1_000_000

shows = (
    spark.range(0, N)
    .withColumn(
        "summary",
        F.concat(
            F.lit("<p>Show "),
            (F.col("id") % 100).cast("string"),
            F.lit("</p>")
        )
    )
    .select("id", "summary")
)

shows.show(5, truncate=False)
print("Rows:", shows.count())

+---+-------------+
|id |summary      |
+---+-------------+
|0  |<p>Show 0</p>|
|1  |<p>Show 1</p>|
|2  |<p>Show 2</p>|
|3  |<p>Show 3</p>|
|4  |<p>Show 4</p>|
+---+-------------+
only showing top 5 rows
Rows: 1000000


In [4]:
def clean_summary(text):
    if text is None:
        return None

    return re.sub(r"<[^>]+>", "", text).strip()

clean_summary_udf = F.udf(
    clean_summary,
    StringType()
)

baseline = shows.withColumn(
    "clean_summary",
    clean_summary_udf(F.col("summary"))
)

In [5]:
optimized = shows.withColumn(
    "clean_summary",
    F.trim(F.regexp_replace(F.col("summary"), r"<[^>]+>", ""))
)

In [6]:
print("Baseline: ")
baseline.select("summary", "clean_summary").show(10, truncate=False)
print()
print("Optimized: ")
optimized.select("summary", "clean_summary").show(10, truncate=False)

Baseline: 
+-------------+-------------+
|summary      |clean_summary|
+-------------+-------------+
|<p>Show 0</p>|Show 0       |
|<p>Show 1</p>|Show 1       |
|<p>Show 2</p>|Show 2       |
|<p>Show 3</p>|Show 3       |
|<p>Show 4</p>|Show 4       |
|<p>Show 5</p>|Show 5       |
|<p>Show 6</p>|Show 6       |
|<p>Show 7</p>|Show 7       |
|<p>Show 8</p>|Show 8       |
|<p>Show 9</p>|Show 9       |
+-------------+-------------+
only showing top 10 rows

Optimized: 
+-------------+-------------+
|summary      |clean_summary|
+-------------+-------------+
|<p>Show 0</p>|Show 0       |
|<p>Show 1</p>|Show 1       |
|<p>Show 2</p>|Show 2       |
|<p>Show 3</p>|Show 3       |
|<p>Show 4</p>|Show 4       |
|<p>Show 5</p>|Show 5       |
|<p>Show 6</p>|Show 6       |
|<p>Show 7</p>|Show 7       |
|<p>Show 8</p>|Show 8       |
|<p>Show 9</p>|Show 9       |
+-------------+-------------+
only showing top 10 rows


In [7]:
def summarize_cleaned(df):
    return (
        df.groupBy("clean_summary")
        .count()
        .orderBy("clean_summary")
    )

baseline_result = summarize_cleaned(baseline)
optimized_result = summarize_cleaned(optimized)

baseline_rows = baseline_result.collect()
optimized_rows = optimized_result.collect()

assert baseline_rows == optimized_rows, "Full-data grouped results differ!"
print("Full-data validation passed.")

Full-data validation passed.


In [8]:
def benchmark(label, df, repeats=5):
    times = []

    for run in range(repeats):
        start = time.perf_counter()
        df.collect()
        elapsed = time.perf_counter() - start

        times.append(elapsed)
        print(f"{label} | run {run + 1}: {elapsed:.3f}s")

    return times

# Warm-up: execute both implementations before recording measurements.
baseline_result.collect()
optimized_result.collect()

# Measured trials:
baseline_times = benchmark(
    "Python UDF",
    baseline_result,
    repeats=5
)

optimized_times = benchmark(
    "Built-in",
    optimized_result,
    repeats=5
)

Python UDF | run 1: 0.106s
Python UDF | run 2: 0.106s
Python UDF | run 3: 0.079s
Python UDF | run 4: 0.149s
Python UDF | run 5: 0.094s
Built-in | run 1: 0.068s
Built-in | run 2: 0.076s
Built-in | run 3: 0.079s
Built-in | run 4: 0.089s
Built-in | run 5: 0.067s


In [9]:
from statistics import median

baseline_median = median(baseline_times)
optimized_median = median(optimized_times)

print(f"Python UDF median: {baseline_median:.3f}s")
print(f"Built-in median:   {optimized_median:.3f}s")

if optimized_median > 0:
    speedup = baseline_median / optimized_median
    improvement = (
        (baseline_median - optimized_median)
        / baseline_median
        * 100
    )

    print(f"Speedup: {speedup:.2f}x")
    print(f"Time reduction: {improvement:.1f}%")

Python UDF median: 0.106s
Built-in median:   0.076s
Speedup: 1.39x
Time reduction: 28.3%


In [10]:
baseline.explain("formatted")
optimized.explain("formatted")

== Physical Plan ==
* Project (4)
+- BatchEvalPython (3)
   +- * Project (2)
      +- * Range (1)


(1) Range [codegen id : 1]
Output [1]: [id#0L]
Arguments: Range (0, 1000000, step=1, splits=Some(2))

(2) Project [codegen id : 1]
Output [2]: [id#0L, concat(<p>Show , cast((id#0L % 100) as string), </p>) AS summary#1]
Input [1]: [id#0L]

(3) BatchEvalPython
Input [2]: [id#0L, summary#1]
Arguments: [clean_summary(summary#1)#15], [pythonUDF0#48]

(4) Project [codegen id : 2]
Output [3]: [id#0L, summary#1, pythonUDF0#48 AS clean_summary#16]
Input [3]: [id#0L, summary#1, pythonUDF0#48]


== Physical Plan ==
* Project (3)
+- * Project (2)
   +- * Range (1)


(1) Range [codegen id : 1]
Output [1]: [id#0L]
Arguments: Range (0, 1000000, step=1, splits=Some(2))

(2) Project [codegen id : 1]
Output [2]: [id#0L, concat(<p>Show , cast((id#0L % 100) as string), </p>) AS summary#1]
Input [1]: [id#0L]

(3) Project [codegen id : 1]
Output [3]: [id#0L, summary#1, trim(regexp_replace(summary#1, <[^>]+>, 